# 🗣️ Group Chat Orchestration for Collaborative Multi-Agent Conversations

## Overview

This notebook demonstrates **Group Chat orchestration** — a multi-agent pattern where an orchestrator coordinates a collaborative conversation among multiple agents, determining speaker selection and conversation flow. This pattern is ideal for scenarios requiring iterative refinement, collaborative problem-solving, or multi-perspective analysis.

Internally, the group chat orchestration assembles agents in a **star topology**, with an orchestrator in the middle. The orchestrator can implement various strategies for selecting which agent speaks next — round-robin, prompt-based selection, or custom logic based on conversation context.

### Key Concepts

| Concept | Description |
|---------|-------------|
| **GroupChatBuilder** | High-level API for building group chat workflows |
| **GroupChatState** | Provides conversation state for selection decisions |
| **Speaker Selection** | Round-robin, agent-based, or custom logic |
| **Iterative Collaboration** | Agents build upon each other's contributions |
| **Context Synchronization** | All agents see the full conversation history |

### Differences Between Group Chat and Other Patterns

| Aspect | Group Chat | Handoff | Sequential |
|--------|-----------|---------|------------|
| **Coordination** | Centralized orchestrator | Agents transfer control directly | Fixed linear order |
| **Refinement** | Iterative, multi-round | Single pass per agent | Single pass per agent |
| **Speaker Selection** | Flexible strategies | Dynamic routing | Predetermined |
| **Context** | Full shared history | Full context handed off | Shared context object |

### Architecture

```
User Task
    ↓
┌─────────────────────────────────────┐
│     Group Chat Orchestrator         │
│  (speaker selection strategy)       │
│         ↓            ↓              │
│   ┌──────────┐ ┌──────────┐        │
│   │Researcher│ │  Writer  │        │
│   └──────────┘ └──────────┘        │
│     Round 1: Researcher gathers     │
│     Round 2: Writer synthesizes     │
│     Round N: Iterative refinement   │
└─────────────────────────────────────┘
    ↓
Final Conversation Output
```

### What You'll Learn

1. How to create specialized agents for group collaboration
2. How to configure a **round-robin** speaker selection strategy
3. How to use an **agent-based orchestrator** for intelligent speaker selection
4. How to implement **custom speaker selection** logic
5. How to stream and process workflow events in real-time

## Prerequisites

- ✅ Azure AI Foundry project configured
- ✅ Environment variables: `AI_FOUNDRY_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- ✅ Azure CLI authentication: Run `az login`

## 1️⃣ Install Dependencies and Import Libraries

In [ ]:
# Install required packages (uncomment if needed)
# %pip install agent-framework azure-identity python-dotenv

import asyncio
import os
from typing import cast, Callable
from collections import OrderedDict
from dataclasses import dataclass, field

from dotenv import load_dotenv
from azure.identity import AzureCliCredential

from agent_framework import (
    Agent,
    AgentResponseUpdate,
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowEvent,
    handler,
)
from agent_framework.foundry import FoundryChatClient

# Load environment variables from .env file
load_dotenv('../../.env')

# Verify environment is loaded
PROJECT_ENDPOINT = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
print(f"✅ Environment loaded: {PROJECT_ENDPOINT is not None and MODEL_DEPLOYMENT is not None}")

## 2️⃣ Set Up the Foundry Chat Client

Initialize the `FoundryChatClient` using `AzureCliCredential` for authentication. This client will be shared across all agents.

In [ ]:
# Create shared Foundry Chat Client
chat_client = FoundryChatClient(
    credential=AzureCliCredential(),
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT,
)

print("✅ Foundry Chat Client created")

## 3️⃣ Define Specialized Agents

Create two agents with distinct, complementary roles:

| Agent | Role | Responsibility |
|-------|------|---------------|
| **Researcher** | Information Gatherer | Collects concise, factual background information |
| **Writer** | Content Synthesizer | Composes clear, structured answers from gathered information |

In [ ]:
# Create a researcher agent
researcher = Agent(
    client=chat_client,
    name="Researcher",
    description="Collects relevant background information.",
    instructions="Gather concise facts that help answer the question. Be brief and factual.",
)

# Create a writer agent
writer = Agent(
    client=chat_client,
    name="Writer",
    description="Synthesizes polished answers using gathered information.",
    instructions="Compose clear, structured answers using any notes provided. Be comprehensive.",
)

print("✅ Researcher Agent created")
print("✅ Writer Agent created")

## 4️⃣ Group Chat with Round-Robin Speaker Selection

The simplest speaker selection strategy is **round-robin** — agents take turns in a fixed order. The `GroupChatState` provides the `current_round` index and `participants` mapping to determine who speaks next.

### How It Works

1. The orchestrator calls the `round_robin_selector` function each round
2. The function picks the next speaker by cycling through participant names
3. The conversation continues until the termination condition is met (4+ messages)

In [ ]:
@dataclass
class GroupChatState:
    """Tracks conversation state for the group chat."""
    conversation: list[Message] = field(default_factory=list)
    current_round: int = 0
    participants: OrderedDict[str, Agent] = field(default_factory=OrderedDict)


class GroupChatCoordinator(Executor):
    """
    A custom executor that coordinates a group chat among multiple agents.
    Supports round-robin, agent-based, or custom speaker selection strategies.
    """

    def __init__(
        self,
        participants: list[Agent],
        select_speaker_func: Callable[[GroupChatState], str] | None = None,
        orchestrator_agent: Agent | None = None,
        termination_condition: Callable[[list[Message]], bool] | None = None,
        max_rounds: int = 10,
        name: str = "GroupChatCoordinator",
    ):
        self.participants_map = OrderedDict((a.name, a) for a in participants)
        self.select_speaker_func = select_speaker_func
        self.orchestrator_agent = orchestrator_agent
        self.termination_condition = termination_condition or (lambda msgs: False)
        self.max_rounds = max_rounds
        self._name = name

    @property
    def name(self) -> str:
        return self._name

    async def _select_speaker_via_agent(self, state: GroupChatState) -> str:
        """Use the orchestrator agent to decide who speaks next."""
        names = list(state.participants.keys())
        prompt = (
            f"Given the conversation so far, select the next speaker from: {names}.\n"
            f"Current round: {state.current_round}.\n"
            f"Reply with ONLY the agent name, nothing else."
        )
        history = list(state.conversation) + [Message(role="user", text=prompt)]
        response = await self.orchestrator_agent.get_response(history)
        selected = response.text.strip()
        # Fuzzy match: pick the best match from participant names
        for name in names:
            if name.lower() in selected.lower():
                print(f"  🤖 Orchestrator selected: '{name}'")
                return name
        # Fallback to first participant
        fallback = names[0]
        print(f"  ⚠️ Orchestrator returned '{selected}', falling back to '{fallback}'")
        return fallback

    @handler
    async def handle(self, context: WorkflowContext, message: Message):
        state = GroupChatState(
            conversation=[message],
            current_round=0,
            participants=self.participants_map,
        )

        for round_num in range(self.max_rounds):
            state.current_round = round_num

            # Select speaker
            if self.select_speaker_func:
                speaker_name = self.select_speaker_func(state)
            elif self.orchestrator_agent:
                speaker_name = await self._select_speaker_via_agent(state)
            else:
                # Default round-robin
                names = list(state.participants.keys())
                speaker_name = names[round_num % len(names)]

            agent = state.participants.get(speaker_name)
            if not agent:
                break

            # Run the selected agent
            response = await agent.get_response(list(state.conversation))

            # Create response message with author info
            agent_msg = Message(role="assistant", text=response.text)
            agent_msg.author_name = speaker_name
            state.conversation.append(agent_msg)

            # Emit streaming update
            await context.emit(WorkflowEvent.data(
                AgentResponseUpdate(text=f"\n[{speaker_name}]: {response.text}")
            ))

            # Check termination
            if self.termination_condition(state.conversation):
                print(f"  ⏹️ Termination condition met after round {round_num}")
                break

        return state.conversation


# --- Round-Robin Selector ---

def round_robin_selector(state: GroupChatState) -> str:
    """A round-robin selector function that picks the next speaker based on the current round index."""
    participant_names = list(state.participants.keys())
    selected = participant_names[state.current_round % len(participant_names)]
    print(f"  🔄 Round {state.current_round}: Selected '{selected}'")
    return selected


# Build the group chat workflow with round-robin selection
coordinator = GroupChatCoordinator(
    participants=[researcher, writer],
    select_speaker_func=round_robin_selector,
    termination_condition=lambda conversation: len(conversation) >= 4,
    name="RoundRobinCoordinator",
)

round_robin_workflow = WorkflowBuilder(start_executor=coordinator).build()

print("✅ Round-robin group chat workflow built")

## 5️⃣ Run the Round-Robin Group Chat

Execute the workflow and process streaming events. The workflow emits:
- **`AgentResponseUpdate`** events during streaming (partial agent output)
- **Final output** event with the complete conversation as `list[ChatMessage]`

In [ ]:
task = "What are the key benefits of async/await in Python?"

print(f"📋 Task: {task}\n")
print("=" * 80)

final_conversation: list[Message] = []
last_executor_id: str | None = None

# Run the round-robin workflow
async for event in round_robin_workflow.run(task, stream=True):
    if event.type == "data" and isinstance(event.data, AgentResponseUpdate):
        # Print streaming agent updates
        eid = event.executor_id
        if eid != last_executor_id:
            if last_executor_id is not None:
                print()
            print(f"\n[{eid}]:", end=" ", flush=True)
            last_executor_id = eid
        print(event.data, end="", flush=True)
    elif event.type == "output":
        # Workflow completed - data is a list of Message
        final_conversation = cast(list[Message], event.data)

if final_conversation:
    print("\n\n" + "=" * 80)
    print("📝 Final Conversation Summary:")
    for msg in final_conversation:
        author = getattr(msg, "author_name", "Unknown")
        text = getattr(msg, "text", str(msg))
        print(f"\n[{author}]")
        print(f"{text[:200]}..." if len(text) > 200 else text)
        print("-" * 80)

print("\n✅ Round-robin workflow completed.")

## 6️⃣ Group Chat with Agent-Based Orchestrator

Instead of a simple selection function, you can use a full **ChatAgent** as the orchestrator. This agent uses its LLM capabilities to intelligently decide which participant should speak next based on the conversation context.

### Advantages of Agent-Based Orchestration

| Feature | Simple Selector | Agent-Based Orchestrator |
|---------|----------------|--------------------------|
| **Logic** | Deterministic rules | LLM-driven decisions |
| **Context Awareness** | Limited (state only) | Full conversation understanding |
| **Adaptability** | Fixed patterns | Dynamic, content-aware |
| **Complexity** | Low | Medium |

In [ ]:
# Create orchestrator agent for intelligent speaker selection
orchestrator_agent = Agent(
    client=chat_client,
    name="Orchestrator",
    description="Coordinates multi-agent collaboration by selecting speakers",
    instructions="""
You coordinate a team conversation to solve the user's task.

Guidelines:
- Start with Researcher to gather information
- Then have Writer synthesize the final answer
- Only finish after both have contributed meaningfully
""",
)

# Build group chat with agent-based orchestrator
agent_coordinator = GroupChatCoordinator(
    participants=[researcher, writer],
    orchestrator_agent=orchestrator_agent,
    # Hard termination: stop after 4 assistant messages (safety net)
    termination_condition=lambda messages: sum(1 for msg in messages if msg.role == "assistant") >= 4,
    name="AgentOrchestratedCoordinator",
)

agent_orchestrated_workflow = WorkflowBuilder(start_executor=agent_coordinator).build()

print("✅ Agent-orchestrated group chat workflow built")

## 7️⃣ Run the Agent-Orchestrated Group Chat

In [ ]:
task = "What are the key benefits of async/await in Python?"

print(f"📋 Task: {task}\n")
print("=" * 80)

final_conversation: list[Message] = []
last_executor_id: str | None = None

# Run the agent-orchestrated workflow
async for event in agent_orchestrated_workflow.run(task, stream=True):
    if event.type == "data" and isinstance(event.data, AgentResponseUpdate):
        eid = event.executor_id
        if eid != last_executor_id:
            if last_executor_id is not None:
                print()
            print(f"\n[{eid}]:", end=" ", flush=True)
            last_executor_id = eid
        print(event.data, end="", flush=True)
    elif event.type == "output":
        final_conversation = cast(list[Message], event.data)

if final_conversation:
    print("\n\n" + "=" * 80)
    print("📝 Final Conversation Summary:")
    for msg in final_conversation:
        author = getattr(msg, "author_name", "Unknown")
        text = getattr(msg, "text", str(msg))
        print(f"\n[{author}]")
        print(f"{text[:200]}..." if len(text) > 200 else text)
        print("-" * 80)

print("\n✅ Agent-orchestrated workflow completed.")

## 8️⃣ Advanced: Custom Speaker Selection Based on Content

You can implement sophisticated selection logic that inspects conversation content, tracks completion signals, and dynamically routes the conversation.

### Strategy

1. Start with **Researcher** to gather facts
2. When Researcher signals completion, switch to **Writer**
3. If Writer needs more info, route back to **Researcher**
4. Terminate when the Writer delivers a final answer

In [ ]:
def smart_selector(state: GroupChatState) -> str:
    """Select speakers based on conversation content and context."""
    conversation = state.conversation

    last_message = conversation[-1] if conversation else None

    # If no messages yet, start with Researcher
    if not last_message:
        print("  🎯 No messages yet → Starting with Researcher")
        return "Researcher"

    last_text = last_message.text.lower() if last_message.text else ""
    last_author = getattr(last_message, "author_name", "")

    # If researcher finished gathering info, switch to writer
    if last_author == "Researcher" and any(
        phrase in last_text for phrase in ["in summary", "to summarize", "key points", "here are the"]
    ):
        print("  🎯 Researcher appears done → Switching to Writer")
        return "Writer"

    # If writer asks for more info, go back to researcher
    if last_author == "Writer" and any(
        phrase in last_text for phrase in ["need more", "additional information", "could you clarify"]
    ):
        print("  🎯 Writer needs more info → Back to Researcher")
        return "Researcher"

    # Default: alternate based on who spoke last
    if last_author == "Researcher":
        print("  🎯 Researcher spoke last → Writer's turn")
        return "Writer"

    print("  🎯 Writer spoke last → Researcher's turn")
    return "Researcher"


# Build the smart-selector workflow
smart_coordinator = GroupChatCoordinator(
    participants=[researcher, writer],
    select_speaker_func=smart_selector,
    termination_condition=lambda conversation: sum(1 for msg in conversation if msg.role == "assistant") >= 4,
    max_rounds=6,
    name="SmartCoordinator",
)

smart_workflow = WorkflowBuilder(start_executor=smart_coordinator).build()

print("✅ Smart-selector group chat workflow built")

## 9️⃣ Run the Smart-Selector Group Chat

In [ ]:
task = "Explain the differences between threads, multiprocessing, and async/await in Python."

print(f"📋 Task: {task}\n")
print("=" * 80)

final_conversation: list[Message] = []
last_executor_id: str | None = None

# Run the smart-selector workflow
async for event in smart_workflow.run(task, stream=True):
    if event.type == "data" and isinstance(event.data, AgentResponseUpdate):
        eid = event.executor_id
        if eid != last_executor_id:
            if last_executor_id is not None:
                print()
            print(f"\n[{eid}]:", end=" ", flush=True)
            last_executor_id = eid
        print(event.data, end="", flush=True)
    elif event.type == "output":
        final_conversation = cast(list[Message], event.data)

if final_conversation:
    print("\n\n" + "=" * 80)
    print("📝 Final Conversation Summary:")
    for msg in final_conversation:
        author = getattr(msg, "author_name", "Unknown")
        text = getattr(msg, "text", str(msg))
        print(f"\n[{author}]")
        print(f"{text[:200]}..." if len(text) > 200 else text)
        print("-" * 80)

print("\n✅ Smart-selector workflow completed.")

## 📝 Key Takeaways

### Group Chat Orchestration Features

| Feature | Description |
|---------|-------------|
| **Centralized Coordination** | Orchestrator manages who speaks next |
| **Iterative Refinement** | Agents review and build on each other's responses |
| **Flexible Selection** | Round-robin, agent-based, or custom logic |
| **Shared Context** | All agents see the full conversation history |
| **Event Streaming** | Real-time `AgentResponseUpdate` events |

### Speaker Selection Strategies Compared

| Strategy | Use Case | Pros | Cons |
|----------|----------|------|------|
| **Round-Robin** | Equal participation needed | Simple, predictable | No content awareness |
| **Agent-Based** | Complex, adaptive conversations | Intelligent, context-aware | Higher latency/cost |
| **Custom Function** | Domain-specific logic | Full control | Requires domain knowledge |

### When to Use Group Chat

| ✅ Good Fit | ❌ Consider Alternatives |
|------------|-------------------------|
| Iterative refinement | Strict sequential processing → **Sequential** |
| Collaborative problem-solving | Independent parallel tasks → **Concurrent** |
| Multi-perspective analysis | Direct agent-to-agent handoffs → **Handoff** |
| Content creation (writer-reviewer) | Complex dynamic planning → **Magentic** |
| Quality assurance workflows | |

### Context Synchronization

Agents in a group chat do **not** share the same session instance. After each agent's turn, the orchestrator broadcasts the response to all other agents, ensuring every participant has the latest context for their next turn.